In [1]:
import warnings
warnings.simplefilter(action='ignore')
import os, joblib
import numpy as np
import pandas as pd
import polars as pl
import kaggle_evaluation.default_inference_server
from catboost import CatBoostRegressor, Pool
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from sklearn.linear_model import RidgeCV
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.ensemble import StackingRegressor
from sklearn.model_selection import train_test_split

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import RobustScaler
from sklearn.preprocessing import MaxAbsScaler
from sklearn.preprocessing import Normalizer

In [2]:
def preprocessing(data, typ, scaler=None, imputer=None):
    import numpy as np
    from sklearn.impute import SimpleImputer
    from sklearn.preprocessing import MinMaxScaler
    
    main_features = ['E1', 'E2', 'E3', 'E4', 'E5', 'E7', 'E8', 'E9', 'E16', 'E17', 'E19', 'E20', 
                     'S1', 'S2', 'S3', 'S4', 'S5', 'S6', 'S8', 'S12', "I2", "P8", "P9", "P10",
                     
                     "M*", "E*", "I*", "P*", "V*", "S*", "D*",
                     
                     "M*_P*_V*", 'M*_E*_S*', 'M*_P*_I*_V*',
                     'V*_M*_E*_I*', 'V*_S*_D*',
                     'E*_I*_V*_D*', 'M*_V*_S*_D*',
                     'P*_V*_S*', 'P*_M*_D*',
                     'M*_E*_P*_S*', 'M*_E*_I*_P*_V*_S*_D*'
                    ]
    
    # Base aggregated features
    data['M*'] = data[[f'M{i}' for i in range(1, 19)]].sum(axis=1).to_numpy()
    data['E*'] = data[[f'E{i}' for i in range(1, 21)]].sum(axis=1).to_numpy()
    data['I*'] = data[[f'I{i}' for i in range(1, 10)]].sum(axis=1).to_numpy()
    data['P*'] = data[['P8', 'P9', 'P10', 'P12', 'P13']].sum(axis=1).to_numpy()
    data['V*'] = data[[f'V{i}' for i in range(1, 14)]].sum(axis=1).to_numpy()
    data['S*'] = data[[f'S{i}' for i in range(1, 13)]].sum(axis=1).to_numpy()
    data['D*'] = data[[f'D{i}' for i in range(1, 10)]].sum(axis=1).to_numpy()

    # ===== M* (Market) Features =====
    data[f'ME_prod_M*_E*'] = data['M*'] * data['E*']
    data[f'ME_ratio_M*_E*'] = data['M*'] / (data['E*'] + 1e-8)
    main_features.extend([f'ME_prod_M*_E*', f'ME_ratio_M*_E*'])
            
    data[f'MI_prod_M*_I*'] = data['M*'] * data['I*']
    data[f'MI_spread_M*_I*'] = data['M*'] - data['I*']
    main_features.extend([f'MI_prod_M*_I*', f'MI_spread_M*_I*'])
        
    data[f'MP_prod_M*_P*'] = data['M*'] * data['P*']
    data[f'MP_ratio_M*_P*'] = data['M*'] / (data['P*'] + 1e-8)
    main_features.extend([f'MP_prod_M*_P*', f'MP_ratio_M*_P*'])
    
    data[f'MV_prod_M*_V*'] = data['M*'] * data['V*']
    main_features.append(f'MV_prod_M*_V*')
        
    data[f'MS_prod_M*_S*'] = data['M*'] * data['S*']
    data[f'MS_weighted_M*_S*'] = data['M*'] * (1 + data['S*'])
    main_features.extend([f'MS_prod_M*_S*', f'MS_weighted_M*_S*'])
    
    # New M* features
    data[f'MD_prod_M*_D*'] = data['M*'] * data['D*']
    data[f'MD_signal_M*_D*'] = data['M*'] * (2 * data['D*'] - 1)
    main_features.extend([f'MD_prod_M*_D*', f'MD_signal_M*_D*'])
    
    data[f'M_square_M*'] = data['M*'] ** 2
    data[f'M_cube_M*'] = data['M*'] ** 3
    data[f'M_sqrt_M*'] = np.sign(data['M*']) * np.sqrt(np.abs(data['M*']))
    data[f'M_momentum_M*_V*'] = data['M*'] * data['V*'] * data['S*']
    main_features.extend([f'M_square_M*', f'M_cube_M*', f'M_sqrt_M*', f'M_momentum_M*_V*'])
    
    data[f'M_acceleration_M*_V*'] = data['M*'] * (data['V*'] ** 2)
    data[f'M_vol_ratio_M*_V*'] = data['M*'] / (data['V*'] + 1e-8)
    data[f'M_vol_adjusted_M*_V*'] = data['M*'] / np.sqrt(data['V*'] + 1e-8)
    data[f'M_sharpe_M*_V*'] = data['M*'] / (data['V*'] + 0.1)
    main_features.extend([f'M_acceleration_M*_V*', f'M_vol_ratio_M*_V*', 
                          f'M_vol_adjusted_M*_V*', f'M_sharpe_M*_V*'])
    
    data[f'M_trend_strength_M*_V*_S*'] = abs(data['M*']) / (data['V*'] + 1e-8) * abs(data['S*'])
    data[f'M_risk_appetite_M*_I*_V*'] = data['M*'] * data['I*'] / (data['V*'] + 1e-8)
    data[f'M_carry_trade_M*_I*'] = data['M*'] * (1 / (data['I*'] + 1e-8))
    main_features.extend([f'M_trend_strength_M*_V*_S*', f'M_risk_appetite_M*_I*_V*', f'M_carry_trade_M*_I*'])
    
    data[f'M_directional_M*'] = np.sign(data['M*']) * (abs(data['M*']) ** 1.5)
    data[f'M_normalized_M*_E*_P*'] = data['M*'] / (abs(data['E*']) + abs(data['P*']) + 1e-8)
    data[f'M_liquidity_M*_V*_I*'] = data['M*'] / (data['V*'] * (data['I*'] + 0.1))
    main_features.extend([f'M_directional_M*', f'M_normalized_M*_E*_P*', f'M_liquidity_M*_V*_I*'])
    
    data[f'M_regime_M*_D*_V*'] = data['M*'] * data['D*'] / (data['V*'] + 1e-8)
    data[f'M_breakout_M*_V*'] = data['M*'] * np.exp(-data['V*'])
    main_features.extend([f'M_regime_M*_D*_V*', f'M_breakout_M*_V*'])
        
    # ===== E* (Economic) Features =====
    data[f'EI_diff_E*_I*'] = data['E*'] - data['I*']
    data[f'EI_prod_E*_I*'] = data['E*'] * data['I*']
    main_features.extend([f'EI_diff_E*_I*', f'EI_prod_E*_I*'])
        
    data[f'EP_prod_E*_P*'] = data['E*'] * data['P*']
    main_features.append(f'EP_prod_E*_P*')
        
    data[f'EV_prod_E*_V*'] = data['E*'] * data['V*']
    data[f'EV_uncertainty_E*_V*'] = abs(data['E*']) * data['V*']
    main_features.extend([f'EV_prod_E*_V*', f'EV_uncertainty_E*_V*'])
        
    data[f'ES_prod_E*_S*'] = data['E*'] * data['S*']
    data[f'ES_diff_E*_S*'] = data['E*'] - data['S*']
    main_features.extend([f'ES_prod_E*_S*', f'ES_diff_E*_S*'])
    
    data[f'ED_prod_E*_D*'] = data['E*'] * data['D*']
    data[f'ED_regime_E*_D*'] = data['E*'] * (1 + 2 * data['D*'])
    main_features.extend([f'ED_prod_E*_D*', f'ED_regime_E*_D*'])
    
    data[f'E_normalized_E*_V*'] = data['E*'] / (data['V*'] + 1e-8)
    data[f'E_risk_adj_E*_V*_S*'] = data['E*'] * data['S*'] / (data['V*'] + 1e-8)
    main_features.extend([f'E_normalized_E*_V*', f'E_risk_adj_E*_V*_S*'])
        
    # ===== I* (Interest Rate) Features =====
    data[f'IP_prod_I*_P*'] = data['I*'] * data['P*']
    main_features.append(f'IP_prod_I*_P*')
    
    data[f'IV_prod_I*_V*'] = data['I*'] * data['V*']
    main_features.append(f'IV_prod_I*_V*')
        
    data[f'IS_prod_I*_S*'] = data['I*'] * data['S*']
    main_features.append(f'IS_prod_I*_S*')
    
    data[f'ID_prod_I*_D*'] = data['I*'] * data['D*']
    data[f'ID_policy_I*_D*'] = data['I*'] * (1 - data['D*'])
    main_features.extend([f'ID_prod_I*_D*', f'ID_policy_I*_D*'])
    
    data[f'I_yield_curve_I*_M*'] = data['I*'] - data['M*']
    data[f'I_real_rate_I*_E*'] = data['I*'] - data['E*']
    main_features.extend([f'I_yield_curve_I*_M*', f'I_real_rate_I*_E*'])
    
    data[f'I_vol_adjusted_I*_V*'] = data['I*'] / (1 + data['V*'])
    main_features.append(f'I_vol_adjusted_I*_V*')
    
    data[f'I_square_I*'] = data['I*'] ** 2
    data[f'I_cube_I*'] = data['I*'] ** 3
    data[f'I_sqrt_I*'] = np.sign(data['I*']) * np.sqrt(np.abs(data['I*']))
    main_features.extend([f'I_square_I*', f'I_cube_I*', f'I_sqrt_I*'])
    
    data[f'I_vol_impact_I*_V*'] = data['I*'] * data['V*'] ** 2
    data[f'I_vol_ratio_I*_V*'] = data['I*'] / (data['V*'] + 1e-8)
    data[f'I_vol_spread_I*_V*'] = abs(data['I*']) - data['V*']
    data[f'I_vol_interaction_I*_V*'] = data['I*'] * np.log1p(data['V*'])
    main_features.extend([f'I_vol_impact_I*_V*', f'I_vol_ratio_I*_V*', 
                          f'I_vol_spread_I*_V*', f'I_vol_interaction_I*_V*'])
    
    data[f'I_market_correlation_I*_M*'] = data['I*'] * data['M*']
    data[f'I_market_divergence_I*_M*'] = data['I*'] / (data['M*'] + 1e-8)
    data[f'I_market_sync_I*_M*_V*'] = data['I*'] * data['M*'] / (data['V*'] + 1e-8)
    main_features.extend([f'I_market_correlation_I*_M*', f'I_market_divergence_I*_M*', 
                          f'I_market_sync_I*_M*_V*'])
    
    data[f'I_policy_stance_I*_E*_V*'] = data['I*'] - data['E*'] - data['V*']
    data[f'I_neutral_rate_I*_E*_M*'] = data['I*'] - (data['E*'] + data['M*']) / 2
    data[f'I_tightening_I*_M*'] = data['I*'] * (1 - data['M*'])
    main_features.extend([f'I_policy_stance_I*_E*_V*', f'I_neutral_rate_I*_E*_M*', 
                          f'I_tightening_I*_M*'])
    
    data[f'I_risk_free_proxy_I*_V*'] = data['I*'] * np.exp(-abs(data['V*']))
    data[f'I_term_premium_I*_V*_M*'] = data['I*'] + data['V*'] * abs(data['M*'])
    data[f'I_convexity_I*'] = data['I*'] ** 2 / (abs(data['I*']) + 1)
    main_features.extend([f'I_risk_free_proxy_I*_V*', f'I_term_premium_I*_V*_M*', 
                          f'I_convexity_I*'])
    
    data[f'I_duration_risk_I*_V*'] = abs(data['I*']) * data['V*']
    data[f'I_easing_signal_I*_M*_V*'] = -data['I*'] * data['M*'] / (data['V*'] + 1e-8)
    data[f'I_regime_shift_I*_D*_V*'] = data['I*'] * data['D*'] * data['V*']
    main_features.extend([f'I_duration_risk_I*_V*', f'I_easing_signal_I*_M*_V*', 
                          f'I_regime_shift_I*_D*_V*'])
        
    # ===== P* (Price/Valuation) Features =====
    data[f'PV_prod_P*_V*'] = data['P*'] * data['V*']
    main_features.append(f'PV_prod_P*_V*')
        
    data[f'PS_prod_P*_S*'] = data['P*'] * data['S*']
    data[f'PS_contrarian_P*_S*'] = -data['P*'] * data['S*']
    main_features.extend([f'PS_prod_P*_S*', f'PS_contrarian_P*_S*'])
    
    data[f'PD_prod_P*_D*'] = data['P*'] * data['D*']
    data[f'PD_value_regime_P*_D*'] = data['P*'] / (1 + data['D*'] + 1e-8)
    main_features.extend([f'PD_prod_P*_D*', f'PD_value_regime_P*_D*'])
    
    data[f'P_momentum_P*_M*'] = data['P*'] * data['M*']
    data[f'P_quality_P*_E*'] = data['P*'] / (abs(data['E*']) + 1e-8)
    main_features.extend([f'P_momentum_P*_M*', f'P_quality_P*_E*'])
    
    data[f'P_risk_premium_P*_I*_V*'] = (data['P*'] - data['I*']) * data['V*']
    main_features.append(f'P_risk_premium_P*_I*_V*')
        
    # ===== V* (Volatility) Features =====
    data[f'VS_prod_V*_S*'] = data['V*'] * data['S*']
    data[f'VS_panic_V*_S*'] = data['V*'] * abs(data['S*'])
    main_features.extend([f'VS_prod_V*_S*', f'VS_panic_V*_S*'])
    
    data[f'VD_prod_V*_D*'] = data['V*'] * data['D*']
    data[f'VD_regime_shift_V*_D*'] = data['V*'] * abs(0.5 - data['D*'])
    main_features.extend([f'VD_prod_V*_D*', f'VD_regime_shift_V*_D*'])
    
    data[f'V_skew_V*_S*_M*'] = data['V*'] * data['S*'] * data['M*']
    data[f'V_term_structure_V*_I*'] = data['V*'] / (data['I*'] + 1e-8)
    main_features.extend([f'V_skew_V*_S*_M*', f'V_term_structure_V*_I*'])
    
    data[f'V_square_V*'] = data['V*'] ** 2
    data[f'V_normalized_V*_M*'] = data['V*'] / (abs(data['M*']) + 1e-8)
    main_features.extend([f'V_square_V*', f'V_normalized_V*_M*'])
    
    data[f'V_cube_V*'] = data['V*'] ** 3
    data[f'V_sqrt_V*'] = np.sqrt(data['V*'])
    data[f'V_log_V*'] = np.log1p(data['V*'])
    data[f'V_exp_V*'] = 1 - np.exp(-data['V*'])
    main_features.extend([f'V_cube_V*', f'V_sqrt_V*', f'V_log_V*', f'V_exp_V*'])
    
    data[f'V_inverse_V*'] = 1 / (data['V*'] + 1e-8)
    data[f'V_standardized_V*_M*'] = data['V*'] / (abs(data['M*']) + 0.1)
    data[f'V_relative_V*_E*'] = data['V*'] / (abs(data['E*']) + 1e-8)
    main_features.extend([f'V_inverse_V*', f'V_standardized_V*_M*', f'V_relative_V*_E*'])
    
    data[f'V_market_stress_V*_M*'] = data['V*'] * abs(data['M*'])
    data[f'V_market_calm_V*_M*'] = data['V*'] / (abs(data['M*']) + 1)
    data[f'V_directional_risk_V*_M*'] = data['V*'] * np.sign(data['M*'])
    data[f'V_asymmetric_V*_M*'] = data['V*'] * (1 + np.sign(data['M*']))
    main_features.extend([f'V_market_stress_V*_M*', f'V_market_calm_V*_M*', 
                          f'V_directional_risk_V*_M*', f'V_asymmetric_V*_M*'])
    
    data[f'V_rate_sensitivity_V*_I*'] = data['V*'] * abs(data['I*'])
    data[f'V_rate_hedge_V*_I*'] = data['V*'] / (abs(data['I*']) + 1e-8)
    data[f'V_policy_uncertainty_V*_I*_M*'] = data['V*'] * abs(data['I*'] - data['M*'])
    main_features.extend([f'V_rate_sensitivity_V*_I*', f'V_rate_hedge_V*_I*', 
                          f'V_policy_uncertainty_V*_I*_M*'])
    
    data[f'V_realized_V*_M*'] = data['V*'] * (data['M*'] ** 2)
    data[f'V_implied_spread_V*_M*'] = data['V*'] - abs(data['M*'])
    data[f'V_vol_of_vol_V*'] = (data['V*'] - data['V*'].rolling(2, min_periods=1).mean()) ** 2
    main_features.extend([f'V_realized_V*_M*', f'V_implied_spread_V*_M*', f'V_vol_of_vol_V*'])
    
    data[f'V_risk_premium_V*_I*_M*'] = (data['V*'] - data['I*']) * abs(data['M*'])
    data[f'V_smile_V*_S*'] = data['V*'] * (data['S*'] ** 2)
    data[f'V_surface_V*_I*_S*'] = data['V*'] * data['I*'] * abs(data['S*'])
    main_features.extend([f'V_risk_premium_V*_I*_M*', f'V_smile_V*_S*', f'V_surface_V*_I*_S*'])
    
    data[f'V_compression_V*_M*_I*'] = data['V*'] / (abs(data['M*']) + abs(data['I*']) + 1e-8)
    data[f'V_expansion_V*_D*'] = data['V*'] * (1 + data['D*']) ** 2
    data[f'V_mean_reversion_V*'] = data['V*'] * (1 / (data['V*'] + 1))
    main_features.extend([f'V_compression_V*_M*_I*', f'V_expansion_V*_D*', f'V_mean_reversion_V*'])
    
    data[f'V_tail_risk_V*_S*_M*'] = data['V*'] * abs(data['S*']) * (1 - np.sign(data['M*']) * np.sign(data['S*']))
    data[f'V_dispersion_V*_P*'] = data['V*'] / (data['P*'] + 1e-8)
    data[f'V_correlation_break_V*_M*_I*'] = data['V*'] * abs(np.sign(data['M*']) - np.sign(data['I*']))
    main_features.extend([f'V_tail_risk_V*_S*_M*', f'V_dispersion_V*_P*', f'V_correlation_break_V*_M*_I*'])
    
    # ===== S* (Sentiment) Features =====
    data[f'SD_prod_S*_D*'] = data['S*'] * data['D*']
    data[f'SD_sentiment_regime_S*_D*'] = data['S*'] * (2 * data['D*'] - 1)
    main_features.extend([f'SD_prod_S*_D*', f'SD_sentiment_regime_S*_D*'])
    
    data[f'S_extreme_S*'] = data['S*'] ** 3
    data[f'S_reversal_S*_M*'] = -data['S*'] * data['M*']
    main_features.extend([f'S_extreme_S*', f'S_reversal_S*_M*'])
    
    data[f'S_fear_greed_S*_V*_P*'] = data['S*'] * data['V*'] / (data['P*'] + 1e-8)
    main_features.append(f'S_fear_greed_S*_V*_P*')
    
    # ===== D* (Dummy/Binary) Features =====
    data[f'D_strength_D*_M*'] = data['D*'] * abs(data['M*'])
    data[f'D_volatility_D*_V*'] = data['D*'] * data['V*']
    main_features.extend([f'D_strength_D*_M*', f'D_volatility_D*_V*'])
    
    data[f'D_sentiment_D*_S*'] = data['D*'] * abs(data['S*'])
    data[f'D_valuation_D*_P*'] = (1 - data['D*']) * data['P*']
    main_features.extend([f'D_sentiment_D*_S*', f'D_valuation_D*_P*'])
    
    # ===== Multi-way Interactions (3+ features) =====
    data[f'MEI_macro_M*_E*_I*'] = data['M*'] * data['E*'] * data['I*']
    data[f'MPV_risk_M*_P*_V*'] = data['M*'] * data['P*'] / (data['V*'] + 1e-8)
    main_features.extend([f'MEI_macro_M*_E*_I*', f'MPV_risk_M*_P*_V*'])
    
    data[f'PVS_sentiment_value_P*_V*_S*'] = data['P*'] * data['V*'] * data['S*']
    data[f'IVS_rate_risk_I*_V*_S*'] = data['I*'] * data['V*'] * abs(data['S*'])
    main_features.extend([f'PVS_sentiment_value_P*_V*_S*', f'IVS_rate_risk_I*_V*_S*'])
    
    data[f'EIVD_stress_E*_I*_V*_D*'] = data['E*'] * data['I*'] * data['V*'] * data['D*']
    data[f'MPSD_complex_M*_P*_S*_D*'] = data['M*'] * data['P*'] * data['S*'] * (1 + data['D*'])
    main_features.extend([f'EIVD_stress_E*_I*_V*_D*', f'MPSD_complex_M*_P*_S*_D*'])
    
    # Advanced M*-I*-V* triple interactions
    data[f'MIV_risk_on_M*_I*_V*'] = data['M*'] * data['I*'] / (data['V*'] + 1e-8)
    data[f'MIV_risk_off_M*_I*_V*'] = data['M*'] / ((data['I*'] + 1e-8) * (data['V*'] + 1e-8))
    data[f'MIV_balanced_M*_I*_V*'] = (data['M*'] + data['I*']) / (data['V*'] + 1e-8)
    data[f'MIV_carry_vol_M*_I*_V*'] = data['M*'] * (data['I*'] ** 2) / (data['V*'] ** 2 + 1e-8)
    main_features.extend([f'MIV_risk_on_M*_I*_V*', f'MIV_risk_off_M*_I*_V*', 
                          f'MIV_balanced_M*_I*_V*', f'MIV_carry_vol_M*_I*_V*'])
    
    data[f'MIV_synchronicity_M*_I*_V*'] = data['M*'] * data['I*'] * data['V*']
    data[f'MIV_divergence_M*_I*_V*'] = abs(data['M*'] - data['I*']) * data['V*']
    data[f'MIV_regime_quality_M*_I*_V*'] = (data['M*'] * data['I*']) / (data['V*'] ** 2 + 1e-8)
    main_features.extend([f'MIV_synchronicity_M*_I*_V*', f'MIV_divergence_M*_I*_V*', 
                          f'MIV_regime_quality_M*_I*_V*'])
    
    # ===== Ratio-based Features =====
    data[f'Risk_reward_M*_V*_I*'] = data['M*'] / (data['V*'] + data['I*'] + 1e-8)
    data[f'Sharpe_proxy_M*_V*'] = data['M*'] / (data['V*'] + 1e-8)
    main_features.extend([f'Risk_reward_M*_V*_I*', f'Sharpe_proxy_M*_V*'])
    
    data[f'Value_momentum_P*_M*'] = data['P*'] / (data['M*'] + 1e-8)
    data[f'Growth_quality_E*_P*'] = data['E*'] / (data['P*'] + 1e-8)
    main_features.extend([f'Value_momentum_P*_M*', f'Growth_quality_E*_P*'])
    
    # Additional M*-I*-V* ratio features
    data[f'Information_ratio_M*_I*_V*'] = (data['M*'] - data['I*']) / (data['V*'] + 1e-8)
    data[f'Sortino_proxy_M*_V*'] = data['M*'] / (np.sqrt(data['V*']) + 1e-8)
    data[f'Calmar_proxy_M*_V*'] = data['M*'] / (np.maximum(data['V*'], 0.1))
    data[f'Treynor_proxy_M*_I*_V*'] = (data['M*'] - data['I*']) / (data['V*'] * abs(data['M*']) + 1e-8)
    main_features.extend([f'Information_ratio_M*_I*_V*', f'Sortino_proxy_M*_V*', 
                          f'Calmar_proxy_M*_V*', f'Treynor_proxy_M*_I*_V*'])
    
    # M*-I*-V* composite indicators
    data[f'MIV_composite_alpha_M*_I*_V*'] = data['M*'] - (data['I*'] * data['V*'])
    data[f'MIV_composite_beta_M*_I*_V*'] = data['M*'] * data['V*'] / (data['I*'] + 1e-8)
    data[f'MIV_risk_adjusted_return_M*_I*_V*'] = (data['M*'] + data['I*']) / (data['V*'] + 0.1)
    data[f'MIV_vol_adjusted_carry_M*_I*_V*'] = data['I*'] * np.exp(-data['V*']) * np.sign(data['M*'])
    main_features.extend([f'MIV_composite_alpha_M*_I*_V*', f'MIV_composite_beta_M*_I*_V*',
                          f'MIV_risk_adjusted_return_M*_I*_V*', f'MIV_vol_adjusted_carry_M*_I*_V*'])

    # Original combined features
    data['M*_P*_V*'] = data['M*'] + data['P*'] + data['V*']
    data['M*_E*_S*'] = data['M*'] + data['E*'] + data['S*'] 
    data['M*_P*_I*_V*'] = data['M*'] + data['P*'] + data['I*'] + data['V*'] 
    data['V*_M*_E*_I*'] = data['V*'] + data['M*'] + data['E*'] + data['I*'] 
    data['V*_S*_D*'] = data['V*'] + data['S*'] + data['D*'] 
    data['E*_I*_V*_D*'] = data['E*'] + data['I*'] + data['V*'] + data['D*']
    data['M*_V*_S*_D*'] = data['M*'] + data['V*'] + data['S*'] + data['D*'] 
    data['P*_V*_S*'] = data['P*'] + data['V*'] + data['S*'] 
    data['P*_M*_D*'] = data['P*'] + data['M*'] + data['D*']
    data['M*_E*_P*_S*'] = data['M*'] + data['E*'] + data['P*'] + data['S*']
    data['M*_E*_I*_P*_V*_S*_D*'] = data['M*'] + data['E*'] + data['I*'] + data['P*'] + data['V*'] + data['S*'] + data['D*']
    
    # New combined features
    data['M*_E*_V*_D*'] = data['M*'] + data['E*'] + data['V*'] + data['D*']
    data['I*_P*_S*_D*'] = data['I*'] + data['P*'] + data['S*'] + data['D*']
    data['M*_I*_V*_S*'] = data['M*'] + data['I*'] + data['V*'] + data['S*']
    main_features.extend(['M*_E*_V*_D*', 'I*_P*_S*_D*', 'M*_I*_V*_S*'])
    
    if typ == "train":
        data = data[main_features + ["forward_returns"]].copy()
        data = data.rename(columns={'forward_returns': 'target'})
    else:
        data = data[main_features].copy()
    
    if 'target' in data.columns:
        data = data.dropna()
    
    feature_cols = [col for col in data.columns if col != 'target']
    
    if typ == "train":
        imputer = SimpleImputer(strategy='mean')
        data[feature_cols] = imputer.fit_transform(data[feature_cols])
    else:
        if imputer is None:
            raise ValueError("Imputer must be provided for test data")
        data[feature_cols] = imputer.transform(data[feature_cols])
    
    if typ == "train":
        scaler = MinMaxScaler()
        data[feature_cols] = scaler.fit_transform(data[feature_cols])
    else:
        if scaler is None:
            raise ValueError("Scaler must be provided for test data")
        data[feature_cols] = scaler.transform(data[feature_cols])
    
    if typ == "train":
        return data, scaler, imputer
    else:
        return data

train = pd.read_csv('/kaggle/input/hull-tactical-market-prediction/train.csv')
train_processed, scaler, imputer = preprocessing(train, "train")

train_x = train_processed.drop(columns=["target"])
train_y = train_processed['target']

In [3]:
train_processed

,E1,E2,E3,E4,E5,E7,E8,E9,E16,E17,...,Calmar_proxy_M*_V*,Treynor_proxy_M*_I*_V*,MIV_composite_alpha_M*_I*_V*,MIV_composite_beta_M*_I*_V*,MIV_risk_adjusted_return_M*_I*_V*,MIV_vol_adjusted_carry_M*_I*_V*,M*_E*_V*_D*,I*_P*_S*_D*,M*_I*_V*_S*,target
6969,0.041781,0.727190,0.649542,0.021277,0.979815,0.894408,0.010199,0.018862,0.780305,0.733879,...,0.666144,0.999246,0.653709,0.071106,0.013816,0.481955,0.203323,0.585143,0.151303,0.001145
6970,0.041500,0.728277,0.650503,0.019640,0.980146,0.894352,0.010233,0.018531,0.780123,0.733868,...,0.667467,0.998669,0.665407,0.071111,0.015503,0.488137,0.231401,0.723706,0.161321,0.004738
6971,0.041219,0.736669,0.657605,0.018003,0.980477,0.894295,0.010266,0.018200,0.779941,0.733856,...,0.664924,0.999612,0.655236,0.071102,0.012924,0.481691,0.231662,0.798388,0.157664,0.006016
6972,0.040938,0.747184,0.666562,0.016367,0.980807,0.894394,0.011356,0.017869,0.776877,0.893087,...,0.660634,0.999777,0.667090,0.071091,0.010009,0.480086,0.232325,0.799361,0.128228,0.001414
6973,0.040657,0.750377,0.669411,0.014730,0.981138,0.888036,0.011387,0.017538,0.776701,0.892525,...,0.662641,0.999730,0.673024,0.071096,0.011591,0.480076,0.227515,0.745093,0.132646,-0.007182
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8985,0.282092,0.654241,0.781596,0.152209,0.331238,0.851819,0.042188,0.914626,0.627247,0.437709,...,0.669879,0.999752,0.746113,0.071119,0.013642,0.491305,0.211699,0.585326,0.111920,0.002457
8986,0.281430,0.656796,0.783496,0.150573,0.330907,0.851816,0.042187,0.914957,0.627242,0.437767,...,0.674149,0.999873,0.755195,0.071130,0.017992,0.492751,0.216753,0.654956,0.158484,0.002312
8987,0.280770,0.660234,0.786076,0.148936,0.330576,0.851813,0.042186,0.915288,0.627200,0.437777,...,0.673970,0.999843,0.761758,0.071124,0.019763,0.497669,0.217999,0.616794,0.133373,0.002891
8988,0.280112,0.664119,0.788992,0.147300,0.330245,0.851793,0.042186,0.915619,0.627159,0.437787,...,0.670596,0.999738,0.755709,0.071118,0.016210,0.495962,0.201554,0.584678,0.087079,0.008310


In [4]:
exist = ['E1', 'E10', 'E11', 'E12', 'E13', 'E14', 'E15', 'E16', 'E17', 'E18',
         'E19', 'E2', 'E20', 'E3', 'E4', 'E5', 'E6', 'E7', 'E8', 'E9', 'S1',
       'S2', 'S3', 'S4', 'S5', 'S6', 'S7', 'S8', 'S9', 'S10', 'S11', 'S12',
       'I2', 'P8', 'P9', 'P10', 'P12', 'P13', 'M*', 'E*', 'I*', 'P*', 'V*',
       'S*', 'D*', 'M*_P*_V*', 'M*_E*_S*', 'M*_P*_I*_V*', 'V*_M*_E*_I*',
       'V*_S*_D*', 'E*_I*_V*_D*', 'M*_V*_S*_D*', 'P*_V*_S*', 'P*_M*_D*',
       'M*_E*_P*_S*', 'M*_E*_I*_P*_V*_S*_D*', 'ME_prod_M*_E*',
       'ME_ratio_M*_E*', 'MI_prod_M*_I*', 'MI_ratio_M*_I*', 'MI_spread_M*_I*',
       'MP_prod_M*_P*', 'MP_ratio_M*_P*', 'MV_ratio_M*_V*', 'MV_prod_M*_V*',
       'MS_prod_M*_S*', 'MS_weighted_M*_S*', 'EI_diff_E*_I*', 'EI_ratio_E*_I*',
       'EI_prod_E*_I*', 'EP_prod_E*_P*', 'EP_ratio_E*_P*', 'EV_prod_E*_V*',
       'EV_uncertainty_E*_V*', 'ES_prod_E*_S*', 'ES_diff_E*_S*',
       'IP_prod_I*_P*', 'IP_discount_I*_P*', 'IV_prod_I*_V*', 'IS_prod_I*_S*',
       'PV_prod_P*_V*', 'PV_stability_P*_V*', 'PS_prod_P*_S*',
       'PS_contrarian_P*_S*', 'VS_prod_V*_S*', 'VS_panic_V*_S*', 'target']

In [5]:
column = [i for i in list(train_processed.columns) if i not in  exist]

In [6]:
['E1', 'E4', 'E5', 'E6', 'E10', 'E11', 'E12', 'E13', 'E14', 'E15', 'E18', 'S7', 'S9', 'S10', 'S11', 'D*', 
 'P12', 'P13', 'MI_ratio_M*_I*', 'IP_discount_I*_P*', 'EP_ratio_E*_P*', 'EI_ratio_E*_I*', 'MV_ratio_M*_V*', 
 'PV_stability_P*_V*']

['E1',
 'E4',
 'E5',
 'E6',
 'E10',
 'E11',
 'E12',
 'E13',
 'E14',
 'E15',
 'E18',
 'S7',
 'S9',
 'S10',
 'S11',
 'D*',
 'P12',
 'P13',
 'MI_ratio_M*_I*',
 'IP_discount_I*_P*',
 'EP_ratio_E*_P*',
 'EI_ratio_E*_I*',
 'MV_ratio_M*_V*',
 'PV_stability_P*_V*']

import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import pearsonr

# Calculate correlations with target
correlations = {}
for col in column:
    if col != 'target':
        corr, p_value = pearsonr(train_processed[col], train_processed['target'])
        correlations[col] = {'correlation': corr, 'p_value': p_value}

# Sort by absolute correlation
sorted_correlations = sorted(correlations.items(), 
                            key=lambda x: abs(x[1]['correlation']), 
                            reverse=True)

# Print correlation summary
print("=" * 60)
print("CORRELATION ANALYSIS WITH TARGET")
print("=" * 60)
for col, stats in sorted_correlations:
    corr = stats['correlation']
    p_val = stats['p_value']
    strength = 'Strong' if abs(corr) > 0.7 else 'Moderate' if abs(corr) > 0.4 else 'Weak'
    print(f"{col:30s} | Corr: {corr:7.4f} | p-value: {p_val:.4e} | {strength}")
print("=" * 60)

# Plot each feature with improved visualization
n_samples = 2000
fig_size = (14, 6)

for col, stats in sorted_correlations:
    print(col)
    if True:
        corr = stats['correlation']
        
        fig, axes = plt.subplots(1, 2, figsize=fig_size)
        
        # Left plot: Time series comparison with scatter plots
        ax1 = axes[0]
        ax1_twin = ax1.twinx()
        
        # Create x-axis (sample indices)
        x_indices = np.arange(n_samples)
        
        # Scatter plot for feature
        scatter1 = ax1.scatter(x_indices, train_processed[col][:n_samples], 
                              color='steelblue', s=50, label=col, alpha=0.7, 
                              edgecolors='darkblue', linewidth=0.5)
        
        # Scatter plot for target
        scatter2 = ax1_twin.scatter(x_indices, train_processed['target'][:n_samples], 
                                   color='coral', s=50, label='target', alpha=0.7,
                                   edgecolors='darkred', linewidth=0.5, marker='s')
        
        ax1.set_xlabel('Sample Index', fontsize=11, fontweight='bold')
        ax1.set_ylabel(col, color='steelblue', fontsize=11, fontweight='bold')
        ax1_twin.set_ylabel('Target', color='coral', fontsize=11, fontweight='bold')
        ax1.tick_params(axis='y', labelcolor='steelblue')
        ax1_twin.tick_params(axis='y', labelcolor='coral')
        ax1.grid(True, alpha=0.3)
        
        # Combined legend
        ax1.legend([scatter1, scatter2], [col, 'target'], loc='upper left')
        
        # Right plot: Scatter plot (feature vs target)
        ax2 = axes[1]
        scatter = ax2.scatter(train_processed[col][:n_samples], 
                             train_processed['target'][:n_samples],
                             alpha=0.6, c=range(n_samples), cmap='viridis', 
                             edgecolors='black', linewidth=0.5, s=60)
        
        # Add regression line
        z = np.polyfit(train_processed[col][:n_samples], 
                       train_processed['target'][:n_samples], 1)
        p = np.poly1d(z)
        x_line = np.linspace(train_processed[col][:n_samples].min(), 
                            train_processed[col][:n_samples].max(), 100)
        ax2.plot(x_line, p(x_line), "r--", linewidth=2, label='Linear fit')
        
        ax2.set_xlabel(col, fontsize=11, fontweight='bold')
        ax2.set_ylabel('Target', fontsize=11, fontweight='bold')
        ax2.legend()
        ax2.grid(True, alpha=0.3)
        
        # Add colorbar
        cbar = plt.colorbar(scatter, ax=ax2)
        cbar.set_label('Sample Index', fontsize=10)
        
        # Overall title
        strength = 'Strong' if abs(corr) > 0.7 else 'Moderate' if abs(corr) > 0.4 else 'Weak'
        fig.suptitle(f'{col} vs Target | Correlation: {corr:.4f} ({strength})', 
                     fontsize=14, fontweight='bold', y=1.02)
        
        plt.tight_layout()
        plt.show()

# Optional: Create a correlation heatmap summary
print("\nTop 10 Most Correlated Features:")
for i, (col, stats) in enumerate(sorted_correlations[:10], 1):
    print(f"{i:2d}. {col:30s} : {stats['correlation']:7.4f}")

In [7]:
LGBM_R_parm = {'boosting_type': 'gbdt', 
               'colsample_bytree': 0.9484106149593443, 
               'learning_rate': 0.1988123373955639, 
               'max_bin': 77, 
               'max_depth': 10, 
               'metric': 'mape', 
               'min_child_samples': 81, 
               'min_data_in_leaf': 21, 
               'n_estimators': 5029, 
               'num_leaves': 42, 
               'objective': 'regression_l1', 
               'reg_alpha': 0.6355835028602363, 
               'reg_lambda': 3.109823217156622, 
               'subsample': 0.7300733288106989, 
               'verbosity': -1}

print(f'LGBMRegressor TRAINING...')
LGBM_R = LGBMRegressor(**LGBM_R_parm)
LGBM_R.fit(train_x, train_y)
joblib.dump(LGBM_R, 'LGBM_R.pkl')

LGBM_R_model = joblib.load('LGBM_R.pkl')

LGBMRegressor TRAINING...


In [8]:
SIGNAL_MULTIPLIER = 400.0
MIN_SIGNAL = 0.0
MAX_SIGNAL = 2.0

def convert_ret_to_signal(ret_value):
    return np.clip(ret_value * SIGNAL_MULTIPLIER + 1, MIN_SIGNAL, MAX_SIGNAL)


def predict(test: pl.DataFrame) -> float:
    try:
        test_df = test.to_pandas()
        test_processed = preprocessing(test_df, 'inference', scaler, imputer)
        
        LGBM_R_y_pred = LGBM_R_model.predict(test_processed)
        signal = convert_ret_to_signal(LGBM_R_y_pred)
        
        return float(signal)
        
    except Exception as e:
        print(f"Prediction error: {e}")
        return 1.0 
    
inference_server = kaggle_evaluation.default_inference_server.DefaultInferenceServer(predict)
if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    inference_server.serve()
else:
    inference_server.run_local_gateway(('/kaggle/input/hull-tactical-market-prediction/',))